# 11 - Code Review Agent

Demonstrates the **orchestrator + worker tools** pattern using `CodeReviewAgent`.

An orchestrator `SimpleAgent` autonomously coordinates three specialist reviewer sub-agents exposed as MCP tools. The LLM decides which reviewers to engage based on the submitted code.

```
code-review-agent  (orchestrator)
  ├─ [tool] review_security     →  security-reviewer    (SimpleAgent)
  ├─ [tool] review_performance  →  performance-reviewer (SimpleAgent)
  └─ [tool] review_readability  →  readability-reviewer (SimpleAgent)
```

The agent class sets up all reviewers and their MCP server internally.

## Setup

Create an `OllamaClient` and pass it to `CodeReviewAgent`.

In [ ]:
from aimu.models.ollama import OllamaClient
from aimu.agents.examples import CodeReviewAgent

model_client = OllamaClient(OllamaClient.MODELS.QWEN_3_5_9B)
agent = CodeReviewAgent(model_client)

print("Orchestrator model:", model_client.model.value)
print("Reviewer tools:", [t.name for t in model_client.mcp_client.list_tools()])

## Sample Code

A short snippet with an SQL injection vulnerability and an O(n²) search loop — two issues that span different review dimensions.

In [ ]:
code_snippet = '''
import sqlite3

def get_user(db_path, username):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    # Fetch user record
    cursor.execute("SELECT * FROM users WHERE name = '" + username + "'")
    return cursor.fetchone()

def find_duplicates(items):
    dupes = []
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            if items[i] == items[j] and items[i] not in dupes:
                dupes.append(items[i])
    return dupes
'''

## Run

Pass the code to `agent.run()`. The orchestrator will call each specialist reviewer, then synthesize findings into a prioritized report.

In [ ]:
result = agent.run(f"Review this Python code:\n\n{code_snippet}")
print(result)

Inspect the orchestrator's message history to see each reviewer call and its response.

In [ ]:
agent.messages

## Streaming

`TOOL_CALLING` chunks reveal which reviewer was called and a preview of their findings; `GENERATING` chunks are the orchestrator's synthesis.

In [ ]:
from aimu.models import StreamPhase

for chunk in agent.run(f"Review this Python code:\n\n{code_snippet}", stream=True):
    if chunk.phase == StreamPhase.TOOL_CALLING:
        reviewer = chunk.content["name"]
        response_preview = chunk.content["response"][:80].replace("\n", " ")
        print(f"\n[tool: {reviewer}]\n  {response_preview}...\n")
    elif chunk.phase == StreamPhase.GENERATING:
        print(chunk.content, end="", flush=True)

## Clean Up

In [ ]:
del agent, model_client